[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/minimind-colab/blob/main/notebooks/05_Alignment.ipynb)

# Module 5: Alignment — DPO & GRPO

In [ ]:
# @title 🔧 Environment Setup (Run this first!)
import os

gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU"
print(f"GPU: {gpu_info[0]}")

if not os.path.exists('minimind-colab'):
    !git clone https://github.com/Boyu-Zhang-UOI/minimind-colab.git
os.chdir('minimind-colab')

!pip install -q transformers==4.46.3 datasets==3.6.0 jinja2==3.1.2 tokenizers matplotlib

import sys
sys.path.insert(0, '.')
print("✅ Setup complete!")

## 📚 Overview

After SFT, the model can follow instructions — but it may still produce undesirable responses.
**Alignment** teaches the model to prefer "better" outputs over "worse" ones.

**What we will cover:**
1. The alignment problem — why SFT alone is not enough
2. DPO (Direct Preference Optimization) — learning from preference pairs
3. DPO loss function — the mathematical core
4. DPO training loop
5. GRPO (Group Relative Policy Optimization) — online RL without a critic
6. The rollout engine — generating and scoring multiple responses
7. Comparing alignment methods

**Key source files:** `trainer/train_dpo.py`, `dataset/lm_dataset.py` (DPODataset), `trainer/train_grpo.py`, `trainer/rollout_engine.py`

## 1. The Alignment Problem

An SFT model can answer questions, but it might:
- Give harmful advice when asked
- Fabricate information confidently
- Prefer verbose responses over concise ones

**Alignment** addresses this by teaching the model from human preferences:
- "This response is better than that one"
- Not "generate exactly this text"

Two main approaches:
- **DPO**: Offline — needs pre-collected preference data (chosen vs rejected)
- **GRPO**: Online — generates its own responses and ranks them

## 2. DPO Data Format

DPO uses **preference pairs** — the same prompt with a "chosen" (good) and "rejected" (bad) response:

```json
{
  "chosen": [
    {"role": "user", "content": "Is the earth flat?"},
    {"role": "assistant", "content": "No, the Earth is roughly spherical..."}
  ],
  "rejected": [
    {"role": "user", "content": "Is the earth flat?"},
    {"role": "assistant", "content": "Yes, many people believe the earth is flat..."}
  ]
}
```

The model learns to increase the probability of "chosen" responses and decrease "rejected" ones.

In [ ]:
import torch
import json
import sys
sys.path.insert(0, '.')
from transformers import AutoTokenizer
from dataset.lm_dataset import DPODataset

tokenizer = AutoTokenizer.from_pretrained('./model')

# Create sample DPO data
dpo_data = [
    {
        "chosen": [
            {"role": "user", "content": "What is 2+2?"},
            {"role": "assistant", "content": "2+2 equals 4."}
        ],
        "rejected": [
            {"role": "user", "content": "What is 2+2?"},
            {"role": "assistant", "content": "I think it might be 5 or maybe 3, I am not sure."}
        ]
    },
    {
        "chosen": [
            {"role": "user", "content": "Is the sky blue?"},
            {"role": "assistant", "content": "Yes, the sky appears blue due to Rayleigh scattering of sunlight."}
        ],
        "rejected": [
            {"role": "user", "content": "Is the sky blue?"},
            {"role": "assistant", "content": "The sky is green."}
        ]
    },
]

data_path = '/tmp/sample_dpo.jsonl'
with open(data_path, 'w') as f:
    for item in dpo_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

dataset = DPODataset(data_path, tokenizer, max_length=128)
sample = dataset[0]

print("=== DPO Sample Structure ===")
for key, val in sample.items():
    print(f"  {key}: shape={val.shape}, dtype={val.dtype}")

print(f"\nChosen mask sum: {sample['mask_chosen'].sum().item()} (tokens to score)")
print(f"Rejected mask sum: {sample['mask_rejected'].sum().item()} (tokens to score)")

## 3. The DPO Loss Function

DPO's key insight: we can learn preferences **without training a separate reward model**.

The loss function:

$$\mathcal{L}_{\text{DPO}} = -\log\sigma\!\left(\beta \cdot \left[(\log\pi_\theta(y_w|x) - \log\pi_\theta(y_l|x)) - (\log\pi_{\text{ref}}(y_w|x) - \log\pi_{\text{ref}}(y_l|x))\right]\right)$$

Where:
- $\pi_\theta$ = policy model (trainable)
- $\pi_{\text{ref}}$ = reference model (frozen SFT checkpoint)
- $y_w$ = chosen (winning) response
- $y_l$ = rejected (losing) response
- $\beta$ = controls preference strength (default 0.1)

**Intuition**: The loss decreases when the policy model increases the gap between chosen and rejected responses *more than the reference model does*.

In [ ]:
import torch.nn.functional as F

def logits_to_log_probs(logits, labels):
    """Convert logits to per-token log probabilities."""
    log_probs = F.log_softmax(logits, dim=2)
    log_probs_per_token = torch.gather(log_probs, dim=2, index=labels.unsqueeze(2)).squeeze(-1)
    return log_probs_per_token

def dpo_loss(ref_log_probs, policy_log_probs, mask, beta=0.1):
    """Compute DPO loss from log probabilities.
    
    First half of batch = chosen, second half = rejected.
    """
    ref_log_probs = (ref_log_probs * mask).sum(dim=1)
    policy_log_probs = (policy_log_probs * mask).sum(dim=1)

    batch_size = ref_log_probs.shape[0]
    # Split chosen and rejected
    chosen_ref = ref_log_probs[:batch_size // 2]
    reject_ref = ref_log_probs[batch_size // 2:]
    chosen_policy = policy_log_probs[:batch_size // 2]
    reject_policy = policy_log_probs[batch_size // 2:]

    # Log probability ratios
    pi_logratios = chosen_policy - reject_policy
    ref_logratios = chosen_ref - reject_ref
    logits = pi_logratios - ref_logratios

    loss = -F.logsigmoid(beta * logits)
    return loss.mean()

# Demonstrate with dummy values
print("=== DPO Loss Demo ===")
dummy_ref = torch.randn(4, 10)    # 2 chosen + 2 rejected
dummy_policy = torch.randn(4, 10)
dummy_mask = torch.ones(4, 10)

loss = dpo_loss(dummy_ref, dummy_policy, dummy_mask, beta=0.1)
print(f"DPO Loss: {loss.item():.4f}")

# Show effect of beta
for beta in [0.05, 0.1, 0.5, 1.0]:
    loss = dpo_loss(dummy_ref, dummy_policy, dummy_mask, beta=beta)
    print(f"  beta={beta:.2f} \u2192 loss={loss.item():.4f}")

## 4. DPO Training Loop

The DPO training loop has two models:
1. **Reference model** — Frozen copy of the SFT model (provides the baseline)
2. **Policy model** — Trainable copy (learns to prefer chosen over rejected)

```
For each batch of (chosen, rejected) pairs:
    1. Concatenate chosen + rejected inputs
    2. Get log-probs from frozen reference model
    3. Get log-probs from trainable policy model
    4. Compute DPO loss
    5. Backpropagate and update policy model
```

In [ ]:
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from torch import optim
from torch.utils.data import DataLoader
from contextlib import nullcontext

config = MiniMindConfig(hidden_size=768, num_hidden_layers=8)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Policy model (trainable)
policy_model = MiniMindForCausalLM(config).to(device)
# Reference model (frozen)
ref_model = MiniMindForCausalLM(config).to(device)
ref_model.load_state_dict(policy_model.state_dict())  # Start from same weights
ref_model.eval()
ref_model.requires_grad_(False)

print(f"Policy model: {sum(p.numel() for p in policy_model.parameters())/1e6:.1f}M params (trainable)")
print(f"Reference model: {sum(p.numel() for p in ref_model.parameters())/1e6:.1f}M params (frozen)")

dataset = DPODataset(data_path, tokenizer, max_length=128)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

optimizer = optim.AdamW(policy_model.parameters(), lr=4e-8)
dtype = torch.bfloat16 if device == 'cuda' and torch.cuda.is_bf16_supported() else torch.float32
autocast_ctx = torch.cuda.amp.autocast(dtype=dtype) if device == 'cuda' else nullcontext()

policy_model.train()
dpo_losses = []

print(f"\nDPO Training: 5 epochs")
print(f"{'='*50}")

for epoch in range(5):
    epoch_loss = 0
    for step, batch in enumerate(loader, 1):
        x_chosen = batch['x_chosen'].to(device)
        x_rejected = batch['x_rejected'].to(device)
        y_chosen = batch['y_chosen'].to(device)
        y_rejected = batch['y_rejected'].to(device)
        mask_chosen = batch['mask_chosen'].to(device)
        mask_rejected = batch['mask_rejected'].to(device)

        x = torch.cat([x_chosen, x_rejected], dim=0)
        y = torch.cat([y_chosen, y_rejected], dim=0)
        mask = torch.cat([mask_chosen, mask_rejected], dim=0)

        with autocast_ctx:
            # Reference model (no gradients)
            with torch.no_grad():
                ref_logits = ref_model(x).logits
            ref_lp = logits_to_log_probs(ref_logits, y)

            # Policy model
            policy_logits = policy_model(x).logits
            policy_lp = logits_to_log_probs(policy_logits, y)

            loss = dpo_loss(ref_lp, policy_lp, mask, beta=0.15)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        epoch_loss += loss.item()
        dpo_losses.append(loss.item())

    print(f"Epoch {epoch+1}/5 | DPO Loss: {epoch_loss/max(step,1):.4f}")

print(f"\n\u2705 DPO training complete!")

## 5. GRPO — Group Relative Policy Optimization

GRPO is an **online** alignment method — it generates its own training data:

1. **Rollout**: Generate N responses per prompt (e.g., N=4)
2. **Reward**: Score each response (length, quality, format)
3. **Advantage**: Normalize rewards within the group: `adv = (reward - mean) / std`
4. **Update**: Policy gradient with clipped surrogate objective + KL penalty

```
Prompt → Generate 4 responses → Score them → Rank → Update policy
         [Response A: 2.1]     ↑ advantage = +0.8
         [Response B: 0.5]     ↓ advantage = -0.4
         [Response C: 1.8]     ↑ advantage = +0.5
         [Response D: 0.3]     ↓ advantage = -0.9
```

**Key advantage over DPO**: No pre-collected preference data needed — the model explores and learns from its own generations.

In [ ]:
import re

def rep_penalty(text, n=3, cap=0.5):
    """Penalize repetitive n-grams."""
    toks = re.findall(r"\w+|[^\w\s]", text.lower())
    grams = [tuple(toks[i:i + n]) for i in range(len(toks) - n + 1)]
    return min(cap, (len(grams) - len(set(grams))) * cap * 2 / len(grams)) if grams else 0.0

def calculate_reward(response):
    """Simple reward function combining multiple signals."""
    reward = 0.0
    # Length reward
    reward += 0.5 if 20 <= len(response.strip()) <= 800 else -0.5
    # Thinking structure
    if '</think>' in response:
        thinking_content = response.split('</think>')[0]
        reward += 1.0 if 20 <= len(thinking_content.strip()) <= 300 else -0.5
    # Repetition penalty
    reward -= rep_penalty(response)
    return reward

# Demo with sample responses
responses = [
    "The answer is 42. This is a well-known reference from The Hitchhiker's Guide to the Galaxy.",
    "I think think think think think the answer might be something maybe perhaps.",
    "<think>\nLet me think about this carefully.\n</think>\nThe answer is 42.",
    "42.",
]

print("=== Reward Scoring Demo ===")
for i, resp in enumerate(responses):
    r = calculate_reward(resp)
    print(f"Response {i}: reward={r:+.2f} | '{resp[:60]}...'")

In [ ]:
import torch

# Simulate GRPO advantage computation
num_prompts = 2
num_generations = 4

# Simulated rewards for 2 prompts × 4 generations each
rewards = torch.tensor([2.1, 0.5, 1.8, 0.3,   # Prompt 1: 4 responses
                        1.5, 2.0, 0.8, 1.2])   # Prompt 2: 4 responses

# Group-relative normalization
grouped = rewards.view(num_prompts, num_generations)
mean_r = grouped.mean(dim=1).repeat_interleave(num_generations)
std_r = grouped.std(dim=1).repeat_interleave(num_generations)
advantages = (rewards - mean_r) / (std_r + 1e-4)

print("=== GRPO Advantage Computation ===")
print(f"{'Prompt':>8} {'Gen':>4} {'Reward':>8} {'Adv':>8}")
print("-" * 35)
for i in range(len(rewards)):
    prompt_idx = i // num_generations
    gen_idx = i % num_generations
    print(f"{'P'+str(prompt_idx):>8} {'G'+str(gen_idx):>4} {rewards[i]:>8.2f} {advantages[i]:>+8.2f}")

## 6. The Rollout Engine

The `rollout_engine.py` provides a `TorchRolloutEngine` that:

1. Takes prompt token IDs as input
2. Generates N completions per prompt using `model.generate()`
3. Computes per-token log-probabilities (needed for policy gradients)
4. Returns a `RolloutResult` with output_ids, completion_ids, log-probs, and decoded text

This abstraction allows swapping in different inference backends (e.g., SGLang for faster serving) without changing the training code.

In [ ]:
from trainer.rollout_engine import compute_per_token_logps

# Demonstrate per-token log probability computation
model_demo = MiniMindForCausalLM(config).to(device).eval()

# Create a small sequence
input_ids = torch.randint(0, config.vocab_size, (1, 20)).to(device)
n_keep = 5  # Compute log-probs for last 5 tokens

with torch.no_grad():
    logps = compute_per_token_logps(model_demo, input_ids, n_keep)

print("=== Per-Token Log Probabilities ===")
print(f"Input sequence length: {input_ids.shape[1]}")
print(f"Tokens to score (n_keep): {n_keep}")
print(f"Log-probs shape: {logps.shape}")
print(f"Log-probs: {logps[0].tolist()}")
print(f"Probabilities: {logps[0].exp().tolist()}")

## 7. Comparing Alignment Methods

| Aspect | DPO | GRPO | PPO |
|--------|-----|------|-----|
| **Data** | Offline (preference pairs) | Online (self-generated) | Online (self-generated) |
| **Reward model** | Not needed | Simple signals | Learned critic |
| **Reference model** | Yes (frozen) | Yes (frozen) | Yes (frozen) |
| **Complexity** | Simple | Medium | High |
| **Compute** | Low | Medium | High |
| **Key file** | `train_dpo.py` | `train_grpo.py` | `train_ppo.py` |

**When to use which:**
- **DPO**: When you have preference data and want a simple, stable method
- **GRPO**: When you want online learning with custom reward signals
- **PPO**: When you need a learned value function for complex reward landscapes

## ✏️ Exercises

1. **Beta sensitivity**: Run DPO with beta values of 0.05, 0.15, and 0.5. How does the loss curve change? What happens to generation quality?

2. **Reward engineering**: Add a new reward signal to the `calculate_reward` function (e.g., penalize responses starting with "I" or reward responses that include specific keywords). How does this change GRPO behavior?

3. **Advantage analysis**: What happens in GRPO when all 4 responses get the exact same reward? Compute the advantages manually and explain the result.

4. **Read `train_ppo.py`**: Compare the PPO training loop with GRPO. What additional components does PPO need (hint: value function / critic network)?

## 📝 Summary

In this module we covered:

- **The alignment problem** — SFT models need preference guidance
- **DPO** — Direct preference optimization from chosen/rejected pairs
- **DPO loss** — $-\log\sigma(\beta \cdot (\pi\_\text{logratios} - \text{ref}\_\text{logratios}))$
- **GRPO** — Group-based policy optimization with online generation
- **Reward signals** — Length, thinking structure, repetition penalty
- **Advantage normalization** — Within-group ranking: `(reward - mean) / std`
- **Rollout engine** — Generate + score + compute log-probs for policy gradients
- **Method comparison** — DPO (simple/offline) vs GRPO (online) vs PPO (complex)

🔜 **Next module:** Agent RL for tool use and knowledge distillation!